## 1. Installation et imports

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu

In [ ]:
import os, re, json, math, time, hashlib, unicodedata, platform, random, gc, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pymupdf          # PyMuPDF (import "fitz" dans les anciennes versions)
import faiss

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
print("python", platform.python_version(), "| pymupdf", pymupdf.__doc__.split()[1] if pymupdf.__doc__ else "?", "| faiss", faiss.__version__)

## 2. Configuration

In [ ]:
PDF_PATH = Path("cnil_guide_securite_personnelle.pdf")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step2_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# La check-list finale (pages 59-62) répète les mesures des fiches sous forme de tableau :
# l'indexer ajoute des doublons et du bruit, sans information nouvelle. Désactivée par défaut.
INDEX_CHECKLIST = False

TOP_K_MAX = 10            # profondeur de recherche évaluée

# Titres officiels des fiches (page « sommaire » du PDF). Servent d'en-têtes contextuels
# et de contrôle croisé avec les titres détectés dans le texte.
FICHE_TITLES = {
    0: "Présentation et avant-propos",
    1: "Piloter la sécurité des données",
    2: "Définir un cadre pour les utilisateurs",
    3: "Impliquer et former les utilisateurs",
    4: "Authentifier les utilisateurs",
    5: "Gérer les habilitations",
    6: "Sécuriser les postes de travail",
    7: "Sécuriser l'informatique mobile",
    8: "Protéger le réseau informatique",
    9: "Sécuriser les serveurs",
    10: "Sécuriser les sites web",
    11: "Encadrer les développements informatiques",
    12: "Protéger les locaux",
    13: "Sécuriser les échanges avec l'extérieur",
    14: "Gérer la sous-traitance",
    15: "Encadrer la maintenance et la fin de vie des matériels et logiciels",
    16: "Tracer les opérations",
    17: "Sauvegarder",
    18: "Prévoir la continuité et la reprise d'activité",
    19: "Gérer les incidents et les violations",
    20: "Analyse de risques",
    21: "Chiffrement, hachage, signature",
    22: "Cloud : informatique en nuage",
    23: "Applications mobiles : conception et développement",
    24: "Intelligence artificielle : conception et apprentissage",
    25: "API : interfaces de programmation applicative",
}
SECTION_LABELS = {
    "intro": "Introduction",
    "precautions": "Les précautions élémentaires",
    "interdits": "Ce qu'il ne faut pas faire",
    "plus_loin": "Pour aller plus loin",
    "notes": "Notes de bas de page",
    "checklist": "Check-list d'évaluation",
}